# 01 — Auditoria, Limpeza, Validação Contábil e Engenharia de Features

Este notebook demonstra **passo a passo** o pré-processamento da **Etapa 1** ([docs/1_preprocessing.md](../docs/1_preprocessing.md)), chamando exclusivamente as **funções puras** implementadas em `src/utils/cleaning.py` e orquestradas por `src/data/preprocessing.py` sobre o dataset bruto `dataset.csv`.

In [1]:
import sys
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATASET_RAW_PATH, ITENS_LIMPOS_PATH
from src.data.preprocessing import executar_preprocessamento
from src.utils.cleaning import (
    item_quantidade_valida,
    validar_desconto_em_moeda,
)

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)

df_bruto = pd.read_csv(DATASET_RAW_PATH)
print(f'Dataset bruto: {df_bruto.shape[0]} linhas x {df_bruto.shape[1]} colunas | {df_bruto["pedido_id"].nunique()} pedidos únicos')
df_bruto.head(5)

Dataset bruto: 3815 linhas x 13 colunas | 3003 pedidos únicos


,pedido_id,pedido_valor_total,pedido_data,pedido_tipo_desconto,pedido_valor_desconto,pedido_subtotal,pedido_vendedor,pedido_metodo_pagamento,item_pedido_nome_produto,item_pedido_quantidade,item_pedido_preco_unitario,item_pedido_user_id,evento_nome
0,a231f36f-be88-492c-b719-23830b22fb4b,20.0,2026-03-09 17:31:54.325-03,%,0.0,20,aa4d1cfe-c6c1-4fd8-8c4b-2b1b9942884c,Dinheiro,Alfajor Preto (Pacote x4),1,20.0,aa4d1cfe-c6c1-4fd8-8c4b-2b1b9942884c,ExpoDireto
1,d47eb20b-e24f-4c70-92f9-9b554eb8f280,15.0,2026-03-10 10:27:12.241-03,%,0.0,15,aa4d1cfe-c6c1-4fd8-8c4b-2b1b9942884c,Dinheiro,Cuca Enrolada de Amendoim,1,15.0,aa4d1cfe-c6c1-4fd8-8c4b-2b1b9942884c,ExpoDireto
2,d3b5b66d-d9e1-4924-b507-144e0e0b9ef9,15.0,2026-03-10 11:17:14.665-03,%,0.0,15,aa4d1cfe-c6c1-4fd8-8c4b-2b1b9942884c,Cartão,Cuca Enrolada de Amendoim,1,15.0,aa4d1cfe-c6c1-4fd8-8c4b-2b1b9942884c,ExpoDireto
3,5b4e4307-89b1-4601-bba3-04be3a33a5ea,13.0,2026-03-10 12:02:08.319-03,%,0.0,13,aa4d1cfe-c6c1-4fd8-8c4b-2b1b9942884c,Cartão,Alfajor Preto,1,6.0,aa4d1cfe-c6c1-4fd8-8c4b-2b1b9942884c,ExpoDireto
4,5b4e4307-89b1-4601-bba3-04be3a33a5ea,13.0,2026-03-10 12:02:08.319-03,%,0.0,13,aa4d1cfe-c6c1-4fd8-8c4b-2b1b9942884c,Cartão,Cri-Cri 200g,1,7.0,aa4d1cfe-c6c1-4fd8-8c4b-2b1b9942884c,ExpoDireto


## Passo 1 — Auditoria das Colunas Inválidas de Vendedor
Verificamos que `pedido_vendedor` e `item_pedido_user_id` são idênticas e refletem apenas 2 logins compartilhados (+ 1 nulo).

In [2]:
iguais = df_bruto['pedido_vendedor'].fillna('<NA>').equals(df_bruto['item_pedido_user_id'].fillna('<NA>'))
print('pedido_vendedor == item_pedido_user_id em 100% das linhas?', iguais)
df_bruto['pedido_vendedor'].value_counts(dropna=False).to_frame('qtd_linhas')

pedido_vendedor == item_pedido_user_id em 100% das linhas? True


,qtd_linhas
pedido_vendedor,
aa4d1cfe-c6c1-4fd8-8c4b-2b1b9942884c,2102
3f621f3e-720d-4c68-b436-eaac939ec515,1712
NaN,1


## Passo 2 — Validação Automatizada de Descontos (`validar_desconto_em_moeda`)
Aplicamos a função pura `validar_desconto_em_moeda(subtotal, valor_desconto, valor_total)` em todas as linhas para provar que `pedido_valor_desconto` já está 100% em moeda (R$), inclusive no único registro marcado como `%` (`0.12` na `EXPOBENTO`).

In [3]:
resultados_desc = [
    validar_desconto_em_moeda(r.pedido_subtotal, r.pedido_valor_desconto, r.pedido_valor_total)
    for r in df_bruto.itertuples(index=False)
]
status_desc = pd.Series([res[2] for res in resultados_desc])
print('Resumo da validação de descontos (todas as linhas):')
print("quantidade de linhas:", len(resultados_desc))
print(status_desc.value_counts(dropna=False).to_frame('qtd_linhas'))

caso_pct = df_bruto[(df_bruto['pedido_tipo_desconto'] == '%') & (df_bruto['pedido_valor_desconto'] != 0)]
print('\nÚnico registro com tipo="%" e desconto != 0:')
caso_pct[['pedido_id', 'evento_nome', 'pedido_subtotal', 'pedido_tipo_desconto', 'pedido_valor_desconto', 'pedido_valor_total']]

Resumo da validação de descontos (todas as linhas):
quantidade de linhas: 3815
             qtd_linhas
ja_em_moeda        3815

Único registro com tipo="%" e desconto != 0:


,pedido_id,evento_nome,pedido_subtotal,pedido_tipo_desconto,pedido_valor_desconto,pedido_valor_total
1536,ca605a42-8d41-4c35-a968-18971c6cc0b0,EXPOBENTO,6,%,0.12,5.88


## Passo 3 — Itens com Quantidade Zero (`item_quantidade_valida`)
Listamos as 2 linhas com `item_pedido_quantidade <= 0` que são removidas por representarem itens trocados por pacote no mesmo pedido.

In [4]:
mask_invalida = [not item_quantidade_valida(q) for q in df_bruto['item_pedido_quantidade']]
pedidos_com_zero = df_bruto.loc[mask_invalida, 'pedido_id'].unique()
print(f'Linhas com quantidade <= 0: {sum(mask_invalida)}')
df_bruto[df_bruto['pedido_id'].isin(pedidos_com_zero)][[
    'pedido_id', 'evento_nome', 'item_pedido_nome_produto', 'item_pedido_quantidade', 'item_pedido_preco_unitario', 'pedido_subtotal'
]]

Linhas com quantidade <= 0: 2


,pedido_id,evento_nome,item_pedido_nome_produto,item_pedido_quantidade,item_pedido_preco_unitario,pedido_subtotal
898,ebbed809-e6f4-4b4a-9850-60eaa218e98b,Expo Afubra,Alfajor Preto (Caixa x16),0,5.000000,20
899,ebbed809-e6f4-4b4a-9850-60eaa218e98b,Expo Afubra,Alfajor Preto (Pacote x4),4,5.000000,20
1770,4680d5c5-3262-4e60-829e-5a9f976edaff,EXPOBENTO,Rapadura de Melado,0,7.000000,20
1771,4680d5c5-3262-4e60-829e-5a9f976edaff,EXPOBENTO,Rapadura de Melado (Pacote x3),3,6.666667,20


## Passo 4 & 5 — Reconciliação Contábil (`139 → 0` erros) e Proteção de Produtos
Executamos o pipeline completo `executar_preprocessamento(df_bruto)` e comparamos **Antes vs. Depois** nos pedidos divergentes da `ExpoDireto` e nas embalagens da `1 Moranguinho`.

In [5]:
df_limpo, audit = executar_preprocessamento(df_bruto)

print('=== AUDITORIA DE RECONCILIAÇÃO CONTÁBIL ===')
print(json.dumps(audit['reconciliacao_contabil'], indent=2, ensure_ascii=False))

# Amostra comparativa Antes vs. Depois em pedidos divergentes da ExpoDireto
df_valido = df_bruto[df_bruto['item_pedido_quantidade'] > 0].copy()
soma_bruta = (df_valido['item_pedido_quantidade'] * df_valido['item_pedido_preco_unitario']).groupby(df_valido['pedido_id']).transform('sum')
ids_div = df_valido.loc[(soma_bruta - df_valido['pedido_subtotal']).abs() >= 0.01, 'pedido_id'].unique()[:5]

print('\nAmostra ANTES (bruto ExpoDireto — quantidade em unidades x preço do pacote):')
display(df_valido[df_valido['pedido_id'].isin(ids_div)][[
    'pedido_id', 'item_pedido_nome_produto', 'item_pedido_quantidade', 'item_pedido_preco_unitario', 'pedido_subtotal'
]])

print('Amostra DEPOIS (reconciliado — 0 erros contábeis):')
display(df_limpo[df_limpo['pedido_id'].isin(ids_div)][[
    'pedido_id', 'produto_com_multiplicador', 'multiplicador', 'qtd_unidades_base', 'qtd_pacotes', 'preco_unitario', 'preco_pacote', 'item_valor_total', 'pedido_subtotal'
]])

=== AUDITORIA DE RECONCILIAÇÃO CONTÁBIL ===
{
  "pedidos_divergentes_antes": 139,
  "pedidos_divergentes_depois": 0,
  "max_erro_absoluto_reais": 0.0,
  "casos_por_linha": {
    "A": 2947,
    "B": 148,
    "C": 86,
    "D": 632
  }
}

Amostra ANTES (bruto ExpoDireto — quantidade em unidades x preço do pacote):


,pedido_id,item_pedido_nome_produto,item_pedido_quantidade,item_pedido_preco_unitario,pedido_subtotal
22,e633a701-71c7-43d2-878a-27b8ee0fad46,Rapadura Assada (Pacote x6),24,25.0,100
25,67b29c42-c395-4951-9b97-8f3e74657d77,Pé-de-Moleque,1,5.0,25
26,67b29c42-c395-4951-9b97-8f3e74657d77,Rapadura de Melado (Pacote x3),3,20.0,25
27,3cf53f32-ec29-45ea-9498-fd9a4aeb1816,Rapadura de Melado (Pacote x3),3,20.0,20
53,3e0e8bf8-a313-4b11-8ab1-48ea524a2ffc,Alfajor Preto (Pacote x4),8,20.0,40
54,9ec2e76d-9312-4420-b010-d702ed9c5753,Rapadura Assada (Pacote x6),6,25.0,30
55,9ec2e76d-9312-4420-b010-d702ed9c5753,Rapadura Assada,1,5.0,30


Amostra DEPOIS (reconciliado — 0 erros contábeis):


,pedido_id,produto_com_multiplicador,multiplicador,qtd_unidades_base,qtd_pacotes,preco_unitario,preco_pacote,item_valor_total,pedido_subtotal
22,e633a701-71c7-43d2-878a-27b8ee0fad46,Rapadura Assada (x6),6,24,4,4.166667,25.0,100.0,100.0
25,67b29c42-c395-4951-9b97-8f3e74657d77,Pé-de-Moleque (x1),1,1,1,5.000000,5.0,5.0,25.0
26,67b29c42-c395-4951-9b97-8f3e74657d77,Rapadura de Melado (x3),3,3,1,6.666667,20.0,20.0,25.0
27,3cf53f32-ec29-45ea-9498-fd9a4aeb1816,Rapadura de Melado (x3),3,3,1,6.666667,20.0,20.0,20.0
53,3e0e8bf8-a313-4b11-8ab1-48ea524a2ffc,Alfajor Preto (x4),4,8,2,5.000000,20.0,40.0,40.0
54,9ec2e76d-9312-4420-b010-d702ed9c5753,Rapadura Assada (x6),6,6,1,4.166667,25.0,25.0,30.0
55,9ec2e76d-9312-4420-b010-d702ed9c5753,Rapadura Assada (x1),1,1,1,5.000000,5.0,5.0,30.0


## Passo 6 — Catálogo Normalizado (`20` Produtos Base / `34` SKUs) e Pedidos Consolidados
Confirmamos que **nenhum produto distinto foi fundido indevidamente** (apenas o sinônimo confirmado do Amendoim de R$ 8,00) e inspecionamos os 18 pedidos sinalizados como `is_pedido_consolidado = True`.

In [6]:
print(f"Produtos base ({df_limpo['produto_base'].nunique()}): {sorted(df_limpo['produto_base'].unique().tolist())}")
print(f"SKUs canônicos ({df_limpo['produto_com_multiplicador'].nunique()}):")
display(df_limpo['produto_com_multiplicador'].value_counts().to_frame('ocorrencias').head(15))

print('Pedidos consolidados / fechamento de caixa (18 pedidos):')
cons = (
    df_limpo[df_limpo['is_pedido_consolidado']]
    .groupby('pedido_id', observed=True)
    .agg(
        evento=('evento_original', 'first'),
        data=('pedido_data', 'first'),
        n_itens=('produto_com_multiplicador', 'count'),
        total_unidades=('qtd_unidades_base', 'sum'),
        subtotal=('pedido_subtotal', 'first'),
        total=('pedido_valor_total', 'first'),
    )
    .sort_values('subtotal', ascending=False)
)
display(cons)

Produtos base (20): ['Alfajor Branco', 'Alfajor Preto', 'Amendoim Temperado', 'Amendoim Temperado 100g', 'Bolacha Cobertura Chocolate', 'Bolacha de Milho', 'Bolacha de manteiga', 'Bolacha de natal', 'Carrapinha', 'Cri-Cri 200g', 'Cuca Alemã', 'Cuca Enrolada de Amendoim', 'Cuca Enrolada de Chocolate', 'Pão branco', 'Pão de Cereais', 'Pão de Milho', 'Pé-de-Moleque', 'Rapadura Assada', 'Rapadura de Melado', 'Rapadura de Melado Grão Moído']
SKUs canônicos (34):


,ocorrencias
produto_com_multiplicador,
Cuca Alemã (x1),1037
Cuca Enrolada de Amendoim (x1),543
Rapadura de Melado (x1),316
Rapadura Assada (x6),299
Rapadura Assada (x1),259
Rapadura de Melado (x3),235
Alfajor Preto (x1),232
Alfajor Preto (x4),171
Cri-Cri 200g (x1),127


Pedidos consolidados / fechamento de caixa (18 pedidos):


,evento,data,n_itens,total_unidades,subtotal,total
pedido_id,,,,,,
e4702a4b-f074-43f1-8895-d756a44b7dda,Feira Polêsine,2026-05-18 14:22:09.591-03:00,6,565,3696.0,3693.0
f738b998-d798-44f0-8014-44056f5ab070,ExpoSerra,2026-05-11 13:46:34.631-03:00,3,147,3150.0,3150.0
66a1d962-7f11-4450-8b0c-d536d344b05f,Feira Polêsine,2026-05-18 14:10:05.440-03:00,3,60,1463.0,1463.0
fa71f80d-a7f1-4278-bfed-721deb6c27f3,Wolksfest Agudo,2026-08-03 21:31:37.652-03:00,5,197,1462.0,1462.0
be7423e0-6b96-45e3-96f1-6e680623ecdc,ExpoSerra,2026-05-11 14:49:18.058-03:00,4,222,953.0,917.0
f37f9f24-d29f-4a14-bfb5-0c94dae4a251,Feira Polêsine,2026-05-18 14:06:49.055-03:00,6,116,941.0,941.0
6cf39e27-1482-48a9-b8fa-3f9d5f3f4029,São Francisco de Assis,2026-10-08 23:09:35.007-03:00,2,49,915.0,0.0
0c13a8f9-d183-44d7-bfa4-04e22c1cebf3,Feira Paraíso,2026-05-20 21:47:41.648-03:00,1,26,650.0,650.0
6abfa089-5dc6-425e-9a05-359bb62144e5,ExpoSerra,2026-05-11 14:04:52.601-03:00,6,68,465.0,465.0


## Passo 7 — Verificação do Artefato Parquet (`data/processed/itens_limpos.parquet`)
Lemos o arquivo `.parquet` salvo em disco para comprovar a preservação integral da tipagem forte e 0 valores nulos.

In [7]:
df_parquet = pd.read_parquet(ITENS_LIMPOS_PATH)
print(f'Parquet carregado: {df_parquet.shape[0]} linhas x {df_parquet.shape[1]} colunas | Nulos totais: {int(df_parquet.isna().sum().sum())}')
schema_df = pd.DataFrame({'coluna': df_parquet.columns, 'dtype': [str(t) for t in df_parquet.dtypes]})
display(schema_df)
df_parquet.head(5)

Parquet carregado: 3813 linhas x 22 colunas | Nulos totais: 0


,coluna,dtype
0,pedido_id,string
1,pedido_data,"datetime64[us, America/Sao_Paulo]"
2,evento_nome,category
3,evento_original,category
4,ponto_venda,category
5,pedido_metodo_pagamento,category
6,metodo_pagamento_grupo,category
7,pedido_subtotal,float64
8,pedido_valor_desconto,float64
9,pedido_valor_total,float64


,pedido_id,pedido_data,evento_nome,evento_original,ponto_venda,pedido_metodo_pagamento,metodo_pagamento_grupo,pedido_subtotal,pedido_valor_desconto,pedido_valor_total,produto_base,tipo_embalagem,produto_com_multiplicador,multiplicador,qtd_unidades_base,qtd_pacotes,preco_unitario,preco_pacote,item_valor_total,is_pedido_gratuito,is_pedido_consolidado,is_pedido_varejo
0,a231f36f-be88-492c-b719-23830b22fb4b,2026-03-09 17:31:54.325-03:00,ExpoDireto,ExpoDireto,Padrão,Dinheiro,Dinheiro,20.0,0.0,20.0,Alfajor Preto,Pacote,Alfajor Preto (x4),4,4,1,5.0,20.0,20.0,False,False,True
1,d47eb20b-e24f-4c70-92f9-9b554eb8f280,2026-03-10 10:27:12.241-03:00,ExpoDireto,ExpoDireto,Padrão,Dinheiro,Dinheiro,15.0,0.0,15.0,Cuca Enrolada de Amendoim,Unidade,Cuca Enrolada de Amendoim (x1),1,1,1,15.0,15.0,15.0,False,False,True
2,d3b5b66d-d9e1-4924-b507-144e0e0b9ef9,2026-03-10 11:17:14.665-03:00,ExpoDireto,ExpoDireto,Padrão,Cartão,Cartão,15.0,0.0,15.0,Cuca Enrolada de Amendoim,Unidade,Cuca Enrolada de Amendoim (x1),1,1,1,15.0,15.0,15.0,False,False,True
3,5b4e4307-89b1-4601-bba3-04be3a33a5ea,2026-03-10 12:02:08.319-03:00,ExpoDireto,ExpoDireto,Padrão,Cartão,Cartão,13.0,0.0,13.0,Alfajor Preto,Unidade,Alfajor Preto (x1),1,1,1,6.0,6.0,6.0,False,False,True
4,5b4e4307-89b1-4601-bba3-04be3a33a5ea,2026-03-10 12:02:08.319-03:00,ExpoDireto,ExpoDireto,Padrão,Cartão,Cartão,13.0,0.0,13.0,Cri-Cri 200g,Unidade,Cri-Cri 200g (x1),1,1,1,7.0,7.0,7.0,False,False,True


---

## Parte 2 — Engenharia de Features e Catálogo Oficial (Etapa 2 — `docs/2_features.md`)
Executamos `executar_engenharia_features(df_parquet)` para gerar `pedidos_cestas.parquet` (com features temporais genéricas e economia unitária) e o catálogo oficial `catalog_metadata.json` (`20` bases / `34` SKUs).

In [8]:
from src.data.features import executar_engenharia_features

df_cestas, catalogo = executar_engenharia_features(df_parquet)
print(f'Tabela consolidada: {df_cestas.shape[0]} linhas x {df_cestas.shape[1]} colunas | Nulos: {int(df_cestas.isna().sum().sum())}')
print(f'Catálogo oficial: {catalogo["n_bases"]} produtos base | {catalogo["n_skus"]} SKUs canônicos')
df_catalogo = pd.DataFrame(catalogo['skus'])
display(df_catalogo[['sku_id', 'display_name', 'produto_base', 'multiplicador', 'preco_unitario_ref', 'preco_pacote_ref', 'economia_unitaria_pct', 'frequencia_varejo']].sort_values('frequencia_varejo', ascending=False).head(15))
display(df_cestas[['pedido_id', 'display_name', 'qtd_pacotes', 'qtd_unidades_base', 'preco_unitario', 'preco_pacote', 'economia_unitaria_pct', 'hora', 'faixa_horaria', 'fim_de_semana', 'is_multi_sku']].head(10))

Tabela consolidada: 3813 linhas x 37 colunas | Nulos: 0
Catálogo oficial: 20 produtos base | 34 SKUs canônicos


,sku_id,display_name,produto_base,multiplicador,preco_unitario_ref,preco_pacote_ref,economia_unitaria_pct,frequencia_varejo
16,16,Cuca Alemã (Unidade),Cuca Alemã,1,25.00,25.0,0.00,1029
19,19,Cuca Enrolada de Amendoim (Unidade),Cuca Enrolada de Amendoim,1,15.00,15.0,0.00,538
30,30,Rapadura de Melado (Unidade),Rapadura de Melado,1,7.00,7.0,0.00,310
29,29,Rapadura Assada (Pacote x6),Rapadura Assada,6,4.17,25.0,16.60,293
27,27,Rapadura Assada (Unidade),Rapadura Assada,1,5.00,5.0,0.00,254
31,31,Rapadura de Melado (Pacote x3),Rapadura de Melado,3,6.67,20.0,4.71,234
4,4,Alfajor Preto (Unidade),Alfajor Preto,1,6.00,6.0,0.00,225
6,6,Alfajor Preto (Pacote x4),Alfajor Preto,4,5.00,20.0,16.67,168
15,15,Cri-Cri 200g (Unidade),Cri-Cri 200g,1,8.00,8.0,0.00,123
8,8,Amendoim Temperado (Unidade),Amendoim Temperado,1,8.00,8.0,0.00,102


,pedido_id,display_name,qtd_pacotes,qtd_unidades_base,preco_unitario,preco_pacote,economia_unitaria_pct,hora,faixa_horaria,fim_de_semana,is_multi_sku
0,0055b17b-e0ca-4884-afc9-309823f6e500,Cuca Enrolada de Amendoim (Unidade),1,1,15.0,15.0,0.0,19,Noite,False,False
1,008eadab-2a7c-46b0-90de-ca9c7fb7c3ab,Cuca Alemã (Unidade),4,4,25.0,25.0,0.0,17,Tarde,False,False
2,009fd014-f883-44bc-a444-c134345e91b3,Cuca Enrolada de Amendoim (Unidade),1,1,15.0,15.0,0.0,13,Tarde,False,False
3,00b6a066-e32d-4704-ab6e-167895bcb7cb,Alfajor Preto (Unidade),1,1,6.0,6.0,0.0,13,Tarde,False,False
4,00d1ef1f-84fd-4b4f-9b15-9e3161a56052,Alfajor Preto (Unidade),2,2,6.0,6.0,0.0,16,Tarde,False,False
5,00d2a9a3-1dcd-4e7b-a8e4-e59c049a1f05,Rapadura Assada (Unidade),1,1,5.0,5.0,0.0,19,Noite,True,False
6,00e3476b-5857-4b41-9ecb-17bd92f0ebcb,Rapadura de Melado (Unidade),2,2,7.0,7.0,0.0,13,Tarde,False,False
7,00ebfcab-b190-47ee-8a7e-5802708ce49a,Cuca Alemã (Unidade),1,1,25.0,25.0,0.0,11,Manhã,False,False
8,00febc08-681b-427b-8d40-c52a249d7127,Alfajor Preto (Unidade),1,1,6.0,6.0,0.0,10,Manhã,False,False
9,01065eff-4fa5-452e-abd7-9af14ecd23a7,Alfajor Preto (Unidade),2,2,6.0,6.0,0.0,15,Tarde,True,False
